# NVDA Intrahour Volatility Dataset Construction

**Author:** Ayooluwa Adelagun

**Date:** March 2026

---

## Overview

This notebook constructs a supervised machine learning dataset for predicting the **direction of hourly intrahour volatility** for NVIDIA Corporation (NVDA).

Intrahour volatility is estimated using the **Root Sum of Squares (RSS) of minute-level log returns** within each trading hour. The target variable is binary: `1` if the current hour's intrahour volatility is greater than the preceding hour's, `0` otherwise.

### Data Sources
- `NVDA_full_1min_adjsplit.txt` — Split-adjusted one-minute OHLCV data from 2000-01-03
- `NVDA_full_1hour_adjsplit.txt` — Split-adjusted hourly OHLCV data from 2000-01-03

### Pipeline Summary
1. Load and preprocess minute-level and hourly price data
2. Compute minute log returns and aggregate to hourly intrahour volatility
3. Align volatility series with hourly OHLCV data
4. Engineer feature variables from hourly price data
5. Construct the binary target variable
6. Apply train-test split, outlier removal, and feature scaling
7. Visualise cleaned data distributions

### Feature Variables
| Feature | Description |
|---|---|
| `Return_Squared` | Squared hourly log return (variance proxy) |
| `Hourly Volatility` | RSS of minute-level log returns (intrahour volatility estimate) |
| `target` | 1 if volatility increased vs prior hour, else 0 |

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

---
## 2. Load & Preprocess Data

Both files have no header row, so column names are assigned manually.

**Minute log return** is computed as:

$$r_{t,i} = \ln\left(\frac{P_{t,i}}{P_{t,i-1}}\right)$$

where $P_{t,i}$ is the closing price at the $i$-th minute within hour $t$.

In [ ]:
# Files have no header row so column names are assigned manually
data_1min = pd.read_csv('NVDA_full_1min_adjsplit.txt', header=None)
data_1min.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

data_1hour = pd.read_csv('NVDA_full_1hour_adjsplit.txt', header=None)
data_1hour.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

# Convert Date columns to datetime for time-based operations
data_1min['Date'] = pd.to_datetime(data_1min['Date'])
data_1hour['Date'] = pd.to_datetime(data_1hour['Date'])

# Cast Close to float before any numeric operations
data_1min['Close'] = data_1min['Close'].astype(float)
data_1hour['Close'] = data_1hour['Close'].astype(float)

# Minute log return: ln(P_t) - ln(P_{t-1})
# .diff() subtracts the previous row, giving the period-over-period log change
data_1min['Minute Log Return'] = np.log(data_1min['Close']).diff()

print(data_1min.head(10))

---
## 3. Compute Intrahour Volatility

Intrahour volatility for each trading hour is estimated as the **Root Sum of Squares (RSS)** of the minute-level log returns within that hour:

$$\sigma_t^{\text{intrahour}} = \sqrt{\sum_{i=1}^{N_t} r_{t,i}^2}$$

where $r_{t,i}$ are the minute log returns and $N_t$ is the number of valid observations in hour $t$. Unlike the intraday estimator, no normalisation by $N_t$ is applied here because each trading hour consistently contains a full set of minute-level observations.

Minute data is resampled to hourly frequency using `.resample('h')`. Non-trading hours producing zero volatility are removed in the next step.

In [ ]:
def calculate_intrahour_volatility(dataframe):
    dataframe = dataframe.copy()
    dataframe = dataframe.set_index('Date')

    # Resample to hourly frequency and compute RSS of minute log returns:
    #   Step 1: square each return and sum within the hour
    #   Step 2: take the square root to get the RSS
    intrahour_volatility = (
        dataframe[['Minute Log Return']]
        .resample('h')
        .apply(lambda x: np.sqrt((x ** 2).sum()))
    )

    intrahour_volatility.reset_index(inplace=True)
    intrahour_volatility.columns = ['Date', 'Hourly Volatility']
    return intrahour_volatility

intrahour_vol_df = calculate_intrahour_volatility(data_1min)
print(intrahour_vol_df.head(10))

---
## 4. Remove Zero-Volatility Hours

Hours where the computed intrahour volatility is exactly zero are removed. These correspond to non-trading periods (overnight, weekends, holidays) that appear in the resampled index but contain no valid minute return data.

In [ ]:
def remove_zero_volatility(dataframe):
    # Exclude hours with zero volatility — these are non-trading periods
    # that appear in the resampled index but contain no valid minute return data
    filtered_dataframe = dataframe[dataframe['Hourly Volatility'] != 0]
    return filtered_dataframe

intrahour_vol_df = remove_zero_volatility(intrahour_vol_df)
print(f'Rows after removing zero-volatility hours: {len(intrahour_vol_df):,}')

---
## 5. Align with Hourly OHLCV Data

The computed hourly volatility series is aligned with the hourly OHLCV data through an inner merge on the datetime index. This retains only timestamps present in both the minute-derived volatility series and the hourly price file, ensuring every observation in the final dataset has a corresponding set of OHLCV features.

In [ ]:
# Set Date as index on both dataframes for alignment
intrahour_vol_df = intrahour_vol_df.set_index('Date')
data_1hour = data_1hour.set_index('Date')

# Cast OHLCV columns to float
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    data_1hour[col] = data_1hour[col].astype(float)

# Inner merge: retain only timestamps present in both datasets
data_merged = data_1hour.join(intrahour_vol_df, how='inner')

print(f'Merged dataset shape: {data_merged.shape}')
print(data_merged.head(10))

---
## 6. Engineer Feature Variables

Two features are derived from the aligned hourly data:

- **Return**: hourly log return computed from consecutive hourly closing prices
- **Return_Squared**: squared hourly log return, a standard proxy for hourly variance

The intrahour feature set is deliberately more parsimonious than the intraday set. Daily-level features such as Range, Volume and VIX are not available at hourly resolution and are therefore excluded.

In [ ]:
# Hourly log return: ln(Close_t) - ln(Close_{t-1})
data_merged['Return'] = np.log(data_merged['Close']).diff()

# Squared return: standard proxy for hourly variance
data_merged['Return_Squared'] = np.square(data_merged['Return'])

# Drop NaNs: removes first row (no prior close for log return)
data_merged = data_merged.dropna()

print(data_merged[['Return', 'Return_Squared', 'Hourly Volatility']].head(10))

---
## 7. Construct Target Variable

The target variable is a binary classification label indicating whether intrahour volatility **increased** relative to the preceding hour:

$$\text{target}_t = \begin{cases} 1 & \text{if } \sigma_t^{\text{intrahour}} > \sigma_{t-1}^{\text{intrahour}} \\ 0 & \text{otherwise} \end{cases}$$

The resulting dataset is approximately balanced between the two classes, which is desirable for classification.

In [ ]:
# Target = 1 if current hour's volatility exceeds the prior hour's, else 0
# .shift(1) shifts the series down by one row so each value is compared to the prior hour
data_merged['target'] = np.where(
    data_merged['Hourly Volatility'] > data_merged['Hourly Volatility'].shift(1), 1, 0
)

# First row has no prior hour, so fill NaN with 0
data_merged['target'] = data_merged['target'].fillna(0)
data_merged = data_merged.dropna()

print(f'Final dataset shape: {data_merged.shape}')
print(data_merged['target'].value_counts())

---
## 8. Finalise Dataset

The dataset is trimmed to start from **2005-01-03** to match the intraday dataset date range and remove the structurally different high-volatility regime observed in NVDA's early years (2000-2004). This ensures both datasets are directly comparable over the same time period.

Columns are then reordered to match the canonical feature ordering used in the modelling pipeline.

In [ ]:
# Filter to start from 2005-01-03 to match the intraday dataset date range
# and remove the structurally different high-volatility 2000-2004 regime
data_merged = data_merged[data_merged.index >= '2005-01-03']

# Select and reorder columns into the canonical feature ordering for the modelling pipeline
data_full = data_merged[['Return_Squared', 'Hourly Volatility', 'target']]
data_full.index.name = 'Date'

print(data_full.head(10))
print(f'\nDate range: {data_full.index.min()} to {data_full.index.max()}')
print(f'Total rows: {len(data_full):,}')
print(f'Target distribution: {data_full["target"].value_counts().to_dict()}')

---
## 9. Train-Test Split

The dataset is split chronologically into a training set comprising the first 80% of observations and a test set comprising the final 20%. A random split would allow the model to observe future data during training, constituting look-ahead bias. All subsequent outlier removal and feature scaling are applied exclusively to the training portion.

In [ ]:
# Chronological 80/20 split — no shuffling to prevent future data leaking into training
splitlimit = int(len(data_full) * 0.8)
training_features = data_full.iloc[:splitlimit].copy()
test_features = data_full.iloc[splitlimit:].copy()

print(f'Total rows:    {len(data_full):,}')
print(f'Training rows: {len(training_features):,}')
print(f'Test rows:     {len(test_features):,}')
print(f'Date range: {data_full.index.min()} to {data_full.index.max()}')

---
## 10. Outlier Removal

Outlier removal is applied to the **training set only**. For each feature, a rolling median is computed using a centred window of 41 observations. Observations where the absolute deviation from the rolling median exceeds two times the median absolute deviation (MAD) of all deviations are removed.

Formally, for feature $x_t$ with rolling median $\tilde{x}_t$:

$$d_t = |x_t - \tilde{x}_t|$$

An observation is flagged and removed if:

$$d_t > 2 \times \text{median}(d_1, d_2, \ldots, d_T)$$

The use of the rolling median and MAD makes the procedure robust to the very outliers it is designed to detect.

In [ ]:
# Outlier detection — 2x MAD approach
# 41-period centred rolling median on Hourly Volatility and Return_Squared
training_features['hourly_volatility_rolling_median'] = (
    training_features['Hourly Volatility'].rolling(window=41, center=True, min_periods=1).median()
)
training_features['return_squared_rolling_median'] = (
    training_features['Return_Squared'].rolling(window=41, center=True, min_periods=1).median()
)

training_features['volatility minus median'] = (
    training_features['Hourly Volatility'] - training_features['hourly_volatility_rolling_median']
).abs()
training_features['return minus median'] = (
    training_features['Return_Squared'] - training_features['return_squared_rolling_median']
).abs()

# 2x threshold based on median of absolute deviations
volatility_outliers_removed = training_features[
    ~(training_features['volatility minus median'] > 2 * training_features['volatility minus median'].median())
]
both_outliers_removed = volatility_outliers_removed[
    ~(volatility_outliers_removed['return minus median'] > 2 * volatility_outliers_removed['return minus median'].median())
]

removed = len(training_features) - len(both_outliers_removed)
pct = removed / len(training_features) * 100
print(f'Rows before outlier removal: {len(training_features):,}')
print(f'Rows after outlier removal:  {len(both_outliers_removed):,}')
print(f'Rows removed: {removed:,} ({pct:.1f}%)')

In [ ]:
X_cleaned = both_outliers_removed[['Return_Squared', 'Hourly Volatility']]
Y_cleaned = both_outliers_removed['target']
data_set_cleaned = both_outliers_removed[['Return_Squared', 'Hourly Volatility', 'target']]

print(f'Cleaned training set shape: {X_cleaned.shape}')
print(data_set_cleaned.head())

---
## 11. Feature Scaling

Min-max scaling is applied to transform each feature to the range $[0, 1]$:

$$x_{t,j}^{\text{scaled}} = \frac{x_{t,j} - x_j^{\min}}{x_j^{\max} - x_j^{\min}}$$

The scaler is **fitted on the cleaned training set only**. The test data is scaled using the same training-set parameters. Fitting the scaler on the test set would constitute look-ahead bias, since in a real forecasting setting the future range of each feature would be unknown.

In [ ]:
# Fit scaler on cleaned training features only
scaler = MinMaxScaler()
training_data_features_scaled = scaler.fit_transform(X_cleaned)
training_data_scaled = pd.DataFrame(
    training_data_features_scaled,
    columns=['Return_Squared', 'Hourly Volatility']
)

print(f'Scaled training features shape: {training_data_features_scaled.shape}')

---
## 12. Visualise Cleaned Data

The plots below compare the original and cleaned training series for both features, and show the distribution of each feature separated by target class.

In [ ]:
# Time-series plot: Hourly Volatility — cleaned vs original
df1 = data_set_cleaned.copy()
df2 = training_features.copy()

df1.index = pd.to_datetime(df1.index)
df2.index = pd.to_datetime(df2.index)

scale_factor = 10

plt.figure(figsize=(12, 5))
plt.plot(df2.index, df2['Hourly Volatility'] * scale_factor,
         color='grey', alpha=0.3, linestyle='-', label='Original Data')
plt.plot(df1.index, df1['Hourly Volatility'] * scale_factor,
         color='purple', linestyle='-', label='Cleaned Data')
plt.gca().tick_params(axis='x', labelsize='x-large')
plt.gca().tick_params(axis='y', labelsize='x-large')
plt.grid(True)
plt.xlabel('Date', size=15)
plt.ylabel('Intrahour Volatility ($10^{-1}$)', size=15)
plt.legend(fontsize=13, loc='upper right')
plt.tight_layout()
# plt.savefig('NVDA_Intrahour_Cleaned_Volatility.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Time-series plot: Return Squared — cleaned vs original
scale_factor_rs = 10000

plt.figure(figsize=(12, 5))
plt.plot(df2.index, df2['Return_Squared'] * scale_factor_rs,
         color='grey', alpha=0.3, linestyle='-', label='Original Data')
plt.plot(df1.index, df1['Return_Squared'] * scale_factor_rs,
         color='purple', linestyle='-', label='Cleaned Data')
plt.gca().tick_params(axis='x', labelsize='x-large')
plt.gca().tick_params(axis='y', labelsize='x-large')
plt.grid(True)
plt.ylim(0, 10)
plt.xlabel('Date', size=15)
plt.ylabel('Return Squared ($10^{-4}$)', size=15)
plt.legend(fontsize=13, loc='upper right')
plt.tight_layout()
# plt.savefig('NVDA_Intrahour_Cleaned_ReturnSquared.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Histogram: Hourly Volatility by target class
scale_factor_hist = 1000

plt.figure(figsize=(8, 5))
df1[df1['target'] == 0]['Hourly Volatility'].multiply(scale_factor_hist).hist(
    alpha=0.5, color='blue', bins=30, label='Decrease'
)
df1[df1['target'] == 1]['Hourly Volatility'].multiply(scale_factor_hist).hist(
    alpha=0.5, color='red', bins=30, label='Increase'
)
plt.xlim(0, 25)
plt.gca().tick_params(axis='x', labelsize='x-large')
plt.gca().tick_params(axis='y', labelsize='x-large')
plt.xlabel('Intrahour Volatility ($10^{-3}$)', size=15)
plt.ylabel('Frequency', size=15)
plt.legend(fontsize=13, loc='upper right')
plt.tight_layout()
# plt.savefig('NVDA_Intrahour_Volatility_Histogram.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Histogram: Return Squared by target class
scale_factor_rs_hist = 1000000

plt.figure(figsize=(8, 5))
df1[df1['target'] == 0]['Return_Squared'].multiply(scale_factor_rs_hist).hist(
    alpha=0.5, color='blue', bins=30, label='Decrease'
)
df1[df1['target'] == 1]['Return_Squared'].multiply(scale_factor_rs_hist).hist(
    alpha=0.5, color='red', bins=30, label='Increase'
)
plt.xlim(0, 25)
plt.ylim(0, 15000)
plt.gca().tick_params(axis='x', labelsize='x-large')
plt.gca().tick_params(axis='y', labelsize='x-large')
plt.xlabel('Return Squared ($10^{-6}$)', size=15)
plt.ylabel('Frequency', size=15)
plt.legend(fontsize=13, loc='upper right')
plt.tight_layout()
# plt.savefig('NVDA_Intrahour_ReturnSquared_Histogram.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

---
## 13. Export to CSV

The final dataset is exported to `NVDA_Intrahour_Volatility_Dataset.csv` with the date as the index.

In [ ]:
# Export final dataset with Date as the index
data_full.to_csv('NVDA_Intrahour_Volatility_Dataset.csv')
print('Saved to NVDA_Intrahour_Volatility_Dataset.csv')
print(f'Columns: {list(data_full.columns)}')
print(f'Shape:   {data_full.shape}')